# 02 — Curate human CB2 Ki measurements

This notebook turns the selected ChEMBL human CB2 Ki activity records into one regression target per retained molecular structure. An **activity record** is one reported measurement; several distinct activity IDs can refer to the same structure. Ki measures binding affinity, not receptor activation, therapeutic benefit, or a cannabis experience.

**Proposed version-1 policy, set before inspecting the results:** RDKit parses and sanitizes each supplied SMILES, then writes canonical isomeric SMILES. We preserve represented charge, tautomers, isotopes, and stereochemistry; we do not remove salts, select a largest fragment, neutralize, or infer unspecified stereochemistry. A molecule with multiple disconnected components or an invalid or wildcard structure goes to structure review. The full canonical isomeric SMILES is the structure key. The default conflict threshold is a measurement-level pKi range of **1.0**. The run uses the configured threshold below and treats ranges within `1e-12` below it as equal to protect tenfold Ki differences from floating-point rounding. The retained target is the median pKi within each structure–assay group, followed by the median of those assay medians. Single-measurement structures remain, with their evidence count visible. These are project defaults for this version, not universal scientific standards.

The configured threshold and tolerance are fixed before processing; this notebook does not tune them to the dataset size or a later model score. Structure canonicalization may change a SMILES string simply by changing atom order. [RDKit parsing and canonical SMILES](https://www.rdkit.org/docs/GettingStartedInPython.html) and the [RDKit Book](https://www.rdkit.org/docs/RDKit_Book.html) describe the mechanics.

**File organization update:** Processed artifacts now use descriptive stage folders and numbered runs. Paths in this notebook and its displayed output were updated; numerical output remains from the recorded execution. This rename did not rerun the analysis. See [the processed-data guide](../data/processed/README.md) for current runs and [the rename record](../data/processed/renaming_manifest.json) for the old-to-new mapping. Future full executions create the next available run number and record dates in the completion manifest. Downstream notebooks keep an explicit input manifest, so rerunning an upstream stage does not silently switch their dataset.

**Accepted-run policy:** Keep the completed baseline and tuning outputs from the accepted notebook version, together with their required upstream inputs. Superseded development runs are removed after verification, regardless of scores. Saved model files use lossless gzip compression. Directory creation still produces a separate candidate run; accepting and pruning runs is a deliberate maintenance step.


## 1. Locate the recorded snapshot and load its manifests

The acquisition and selection manifests identify one completed handoff. The selected CSV is resolved beside its selection manifest, while raw paths are resolved from the repository root. This avoids mixing a selected CSV from one download with raw records from another. The notebook works when started from the repository root or `notebooks/`.

In [1]:
# Import tools for paths, saved records, calculations, and run history.
# Pandas handles tables, RDKit reads structures, and openpyxl writes review workbooks.
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import json
import math
import platform
import subprocess

import pandas as pd
import rdkit
from rdkit import Chem
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font

# Set the disagreement cutoff before processing: one pKi unit means a tenfold Ki difference.
# The tiny tolerance handles numerical rounding; the reason label follows the chosen cutoff.
CONFLICT_RANGE_PKI = 1.0
CONFLICT_RANGE_ABS_TOL = 1e-12
CONFLICT_REVIEW_REASON = f"pki_range_ge_{CONFLICT_RANGE_PKI}"
assert CONFLICT_RANGE_PKI > 0 and CONFLICT_RANGE_ABS_TOL > 0
SELECTION_MANIFEST_REL = Path(
    "data/interim/chembl_cb2_20260922T165756862384Z/selection_manifest.json"
)
NOTEBOOK_REL = "notebooks/02_data_curation.ipynb"
run_started_at_utc = datetime.now(timezone.utc).isoformat()

# Find the repository from either its root or notebooks folder, then load the selected run.
# An assertion stops execution if the expected folder or manifest is missing.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
assert (project_root / "notebooks").is_dir(), "Run from the repository root or notebooks folder."
selection_manifest_path = project_root / SELECTION_MANIFEST_REL
assert selection_manifest_path.is_file(), f"Recorded selection is missing: {selection_manifest_path}"
with selection_manifest_path.open(encoding="utf-8") as file:
    selection_manifest = json.load(file)

# Follow the manifest paths to the selected CSV and its original download.
# Check that these files belong to the same acquisition so different runs are not mixed.
selected_path = selection_manifest_path.parent / selection_manifest["selected_file"]
raw_folder = project_root / selection_manifest["source_raw_folder"]
acquisition_manifest_path = project_root / selection_manifest["source_manifest"]
activity_path = raw_folder / "chembl_cb2_activity.json"
assay_path = raw_folder / "chembl_cb2_assay.json"
for path in (selected_path, acquisition_manifest_path, activity_path, assay_path):
    assert path.is_file(), f"Required source file is missing: {path}"
assert acquisition_manifest_path.parent.resolve() == raw_folder.resolve()
assert raw_folder.name == selection_manifest_path.parent.name

# Read the acquisition record and the original activity and assay tables into memory.
# Activity records contain measurements; assay records describe the experiments behind them.
with acquisition_manifest_path.open(encoding="utf-8") as file:
    acquisition_manifest = json.load(file)
with activity_path.open(encoding="utf-8") as file:
    raw_activities = json.load(file)
with assay_path.open(encoding="utf-8") as file:
    raw_assays = json.load(file)

# Verify the human CB2 target, database release, Ki units, record counts, and expected columns.
# Only after these checks pass do we print which source snapshot this run is using.
assert acquisition_manifest["target_chembl_id"] == "CHEMBL253"
assert acquisition_manifest["organism"] == "Homo sapiens"
assert acquisition_manifest["uniprot"] == "P34972"
assert selection_manifest["chembl_version"] == acquisition_manifest["chembl_version"]
assert selection_manifest["ki_unit"] == "nM"
assert len(raw_activities) == acquisition_manifest["activity_records"]
assert len(raw_assays) == acquisition_manifest["assay_records"]
source_columns = [
    "activity_id", "molecule_chembl_id", "canonical_smiles", "ki_nm",
    "assay_chembl_id", "document_chembl_id", "assay_description",
]
assert selection_manifest["columns"] == source_columns
print(f"{'Recorded selection':<24} | {SELECTION_MANIFEST_REL}")
print(f"{'Raw activity records':<24} | {len(raw_activities):>8,}")
print(f"{'Raw assay records':<24} | {len(raw_assays):>8,}")
print(f"{'ChEMBL release':<24} | {acquisition_manifest['chembl_version']}")

Recorded selection       | data/interim/chembl_cb2_20260922T165756862384Z/selection_manifest.json
Raw activity records     |   22,523
Raw assay records        |    1,551
ChEMBL release           | ChEMBL_37


## 2. Verify every selected source value

The selected CSV keeps its seven original columns. Activity IDs and Ki values are parsed deliberately, and blank document IDs remain missing. Each selected activity must match one raw activity and one assay record. A failed comparison stops the notebook because curation must not silently repair a broken upstream handoff.

In [2]:
# Treat whitespace-only strings as missing values when comparing source fields.
# This matches the text cleanup already used in notebook 01.
def blank_to_none(value):
    return (value.strip() or None) if isinstance(value, str) else value


# Read every CSV field as text first, then deliberately convert activity IDs and Ki values.
# Check the schema, unique IDs, positive finite Ki values, and required nonblank fields.
selected = pd.read_csv(selected_path, dtype=str, keep_default_na=False)
assert list(selected.columns) == source_columns, "Selected CSV schema changed."
assert len(selected) == selection_manifest["selected_records"]
assert selected["activity_id"].str.fullmatch(r"[0-9]+").all()
selected["activity_id"] = selected["activity_id"].astype("int64")
selected["ki_nm"] = pd.to_numeric(selected["ki_nm"], errors="raise")
assert selected["activity_id"].is_unique, "Selected activity IDs are repeated."
assert selected["ki_nm"].map(math.isfinite).all() and selected["ki_nm"].gt(0).all()
for field in ("molecule_chembl_id", "canonical_smiles", "assay_chembl_id"):
    assert selected[field].map(blank_to_none).notna().all(), f"Blank required field: {field}"
# Keep missing document IDs as missing rather than dropping those measurements.
# A document ID was not required by the earlier selection rules.
selected["document_chembl_id"] = selected["document_chembl_id"].map(blank_to_none)

# Check raw IDs for duplicates before building dictionaries keyed by those IDs.
# These dictionaries let each selected row find its original measurement and assay directly.
raw_activity_ids = [row["activity_id"] for row in raw_activities]
raw_assay_ids = [row["assay_chembl_id"] for row in raw_assays]
assert len(raw_activity_ids) == len(set(raw_activity_ids)), "Raw activity ID is repeated."
assert len(raw_assay_ids) == len(set(raw_assay_ids)), "Raw assay ID is repeated."
activities_by_id = {row["activity_id"]: row for row in raw_activities}
assays_by_id = {row["assay_chembl_id"]: row for row in raw_assays}

# Compare each selected record with its raw sources, including the assay description and Ki.
# A mismatch stops the notebook instead of silently repairing or removing the affected row.
for row in selected.itertuples(index=False):
    activity = activities_by_id.get(row.activity_id)
    assert activity is not None, f"Activity {row.activity_id} is absent from the raw snapshot."
    assay = assays_by_id.get(row.assay_chembl_id)
    assert assay is not None, f"Activity {row.activity_id} has no raw assay."
    assert blank_to_none(row.molecule_chembl_id) == blank_to_none(activity.get("molecule_chembl_id")), row.activity_id
    assert blank_to_none(row.canonical_smiles) == blank_to_none(activity.get("canonical_smiles")), row.activity_id
    assert blank_to_none(row.assay_chembl_id) == blank_to_none(activity.get("assay_chembl_id")), row.activity_id
    assert row.document_chembl_id == blank_to_none(activity.get("document_chembl_id")), row.activity_id
    assert blank_to_none(row.assay_description) == blank_to_none(assay.get("description")), row.activity_id
    assert math.isclose(row.ki_nm, float(activity["standard_value"]), rel_tol=0, abs_tol=1e-12), row.activity_id

    # Recheck the selection contract: exact human CB2 binding Ki, accepted quality, and no duplicate flag.
    # The final messages are reached only if every selected activity passes all these checks.
    assert activity.get("target_chembl_id") == "CHEMBL253", row.activity_id
    assert activity.get("standard_type") == "Ki", row.activity_id
    assert blank_to_none(activity.get("standard_relation")) == "=", row.activity_id
    assert blank_to_none(activity.get("standard_units")) == "nM", row.activity_id
    assert activity.get("assay_type") == "B" and assay.get("assay_type") == "B", row.activity_id
    assert assay.get("target_chembl_id") == "CHEMBL253", row.activity_id
    assert assay.get("confidence_score") == 9, row.activity_id
    assert assay.get("assay_organism") == "Homo sapiens", row.activity_id
    assert blank_to_none(activity.get("data_validity_comment")) in (None, "Manually validated"), row.activity_id
    assert str(activity.get("potential_duplicate")) == "0", row.activity_id

print(f"Verified selected activities and all seven source fields: {len(selected):,}")
print(f"Missing document IDs retained as missing: {selected['document_chembl_id'].isna().sum():,}")

Verified selected activities and all seven source fields: 4,081
Missing document IDs retained as missing: 0


## 3. Describe the selected measurements

These counts show how many activity records enter curation and how much source metadata they cover. A ChEMBL molecule ID is useful for tracing records, while the validated RDKit SMILES will be the aggregation key. Repeated activity IDs have already been ruled out; multiple activities for one molecule ID may be distinct measurements.

In [3]:
# Group activities by ChEMBL molecule ID and count how many measurements each ID has.
# Several measurements for one molecule do not necessarily mean duplicate experiments.
molecule_activity_counts = selected.groupby("molecule_chembl_id").size()
# Collect the total rows and distinct molecule, assay, and document IDs.
# The repeated-molecule count tells us how often we may have measurements to compare.
selection_counts = {
    "Measurements": len(selected),
    "Molecule IDs": selected["molecule_chembl_id"].nunique(),
    "Assays": selected["assay_chembl_id"].nunique(),
    "Nonmissing documents": selected["document_chembl_id"].nunique(dropna=True),
    "Molecule IDs with multiple activities": int(molecule_activity_counts.gt(1).sum()),
}
# Print a heading with fixed column widths so the different counts line up.
print(f"{'Selected-data count':<40} | {'Value':>9}")
# Display each named count with thousands separators.
# This describes the input table without filtering or changing any measurements.
for label, value in selection_counts.items():
    print(f"{label:<40} | {value:>9,}")

Selected-data count                      |     Value
Measurements                             |     4,081
Molecule IDs                             |     3,637
Assays                                   |       328
Nonmissing documents                     |       281
Molecule IDs with multiple activities    |       345


## 4. Parse each distinct SMILES and assign a structure key

RDKit parses each distinct input string once, with sanitization enabled. A failed parse, zero-atom structure, wildcard atom, or disconnected molecule is assigned a specific review reason; we do not guess which fragment is the ligand. For eligible single-component molecules, `rdkit_smiles` is canonical isomeric SMILES. An unspecified potential stereocenter is a quality flag, not an exclusion.

In [4]:
# Process each distinct source SMILES once and store its result in a lookup dictionary.
# RDKit sanitization checks whether the represented atoms and bonds can be interpreted.
structure_results = {}
for input_smiles in selected["canonical_smiles"].unique():
    result = {
        "rdkit_smiles": None,
        "component_count": None,
        "quality_flags": "",
        "review_reason": "",
    }
    flags = []
    try:
        molecule = Chem.MolFromSmiles(input_smiles, sanitize=True)
    except (ValueError, RuntimeError) as error:
        molecule = None
        flags.append("parse_exception_" + type(error).__name__)

    # Separate failed structures from readable ones, then check for empty, wildcard, or disconnected structures.
    # The ordered conditions choose one primary review reason while keeping any additional flags.
    if molecule is None:
        result["review_reason"] = "parse_or_sanitization_failed"
    else:
        result["component_count"] = len(Chem.GetMolFrags(molecule))
        if molecule.GetNumAtoms() == 0:
            flags.append("zero_atoms")
        if any(atom.GetAtomicNum() == 0 for atom in molecule.GetAtoms()):
            flags.append("wildcard_atom")
        if result["component_count"] > 1:
            flags.append("multiple_components")

        if "zero_atoms" in flags:
            result["review_reason"] = "zero_atoms"
        elif "wildcard_atom" in flags:
            result["review_reason"] = "wildcard_atom"
        elif "multiple_components" in flags:
            result["review_reason"] = "multiple_components_review"
        else:
            try:
                # Write a consistent structure key while preserving represented stereochemistry and isotope information.
                # Flag potential unspecified stereochemistry for visibility; that flag alone does not exclude a row.
                result["rdkit_smiles"] = Chem.MolToSmiles(
                    molecule, canonical=True, isomericSmiles=True
                )
                stereo_info = Chem.FindPotentialStereo(molecule)
                if any(info.specified == Chem.StereoSpecified.Unspecified for info in stereo_info):
                    flags.append("potential_unspecified_stereo")
            except (ValueError, RuntimeError) as error:
                result["review_reason"] = "structure_processing_failed"
                flags.append("processing_exception_" + type(error).__name__)

    result["quality_flags"] = ";".join(flags)
    structure_results[input_smiles] = result

# Copy the selected table and attach each SMILES result to all activities that used it.
# The original SMILES remains available alongside the new structure key and review information.
measurements = selected.copy()
for field in ("rdkit_smiles", "component_count", "quality_flags", "review_reason"):
    measurements[field] = measurements["canonical_smiles"].map(
        {smiles: result[field] for smiles, result in structure_results.items()}
    )

# Read each eligible key back into RDKit and write it again.
# An identical result confirms that the representation is stable in this RDKit version.
eligible_keys = measurements.loc[measurements["review_reason"].eq(""), "rdkit_smiles"].unique()
for smiles in eligible_keys:
    reparsed = Chem.MolFromSmiles(smiles, sanitize=True)
    assert reparsed is not None
    assert Chem.MolToSmiles(reparsed, canonical=True, isomericSmiles=True) == smiles

# Summarize eligible structures, changed strings, ID mappings, and stereochemistry flags.
# Changed atom ordering can change a SMILES string without changing the represented molecule.
eligible = measurements[measurements["review_reason"].eq("")].copy()
changed_string_count = int(eligible["canonical_smiles"].ne(eligible["rdkit_smiles"]).sum())
molecule_to_keys = eligible.groupby("molecule_chembl_id")["rdkit_smiles"].nunique()
key_to_molecules = eligible.groupby("rdkit_smiles")["molecule_chembl_id"].nunique()
print(f"{'Distinct input SMILES':<42} | {len(structure_results):>8,}")
print(f"{'Eligible structure keys':<42} | {len(eligible_keys):>8,}")
print(f"{'Changed SMILES strings (activity rows)':<42} | {changed_string_count:>8,}")
print(f"{'Molecule IDs with multiple structure keys':<42} | {molecule_to_keys.gt(1).sum():>8,}")
print(f"{'Structure keys with multiple molecule IDs':<42} | {key_to_molecules.gt(1).sum():>8,}")
print(f"{'Potential unspecified stereo (activity rows)':<42} | {measurements['quality_flags'].str.contains('potential_unspecified_stereo').sum():>8,}")
if molecule_to_keys.gt(1).any():
    print("Molecule IDs with multiple keys:", molecule_to_keys[molecule_to_keys.gt(1)].head(5).to_dict())
if key_to_molecules.gt(1).any():
    print("Structure keys with multiple molecule IDs:", key_to_molecules[key_to_molecules.gt(1)].head(5).to_dict())
print("\nChanged-string examples; a changed string alone does not imply changed chemistry:")
for row in eligible.loc[eligible["canonical_smiles"].ne(eligible["rdkit_smiles"])].head(3).itertuples():
    print(f"  activity {row.activity_id}: {row.canonical_smiles[:75]} -> {row.rdkit_smiles[:75]}")

Distinct input SMILES                      |    3,637
Eligible structure keys                    |    3,606
Changed SMILES strings (activity rows)     |        2
Molecule IDs with multiple structure keys  |        0
Structure keys with multiple molecule IDs  |        0
Potential unspecified stereo (activity rows) |      892

Changed-string examples; a changed string alone does not imply changed chemistry:
  activity 12720253: O/N=C1C(=C/c2cn(Cc3ccc(Cl)cc3)c3ccccc23)/N2CCC/1CC2 -> O/N=C1/C(=C/c2cn(Cc3ccc(Cl)cc3)c3ccccc23)N2CCC1CC2
  activity 12720254: O/N=C1C(=C/c2cn(Cc3ccccc3)c3ccccc23)/N2CCC/1CC2 -> O/N=C1/C(=C/c2cn(Cc3ccccc3)c3ccccc23)N2CCC1CC2


## 5. Convert Ki to pKi without altering Ki

pKi is the negative base-10 logarithm of molar Ki. With nanomolar input, `pki = 9 - log10(ki_nm)`: a larger pKi means a lower Ki and stronger measured binding affinity. All selected activities receive pKi, including those later sent to structure review, because their numeric Ki already passed the input contract. The raw Ki remains in its own column.

In [5]:
# Check three known conversions before applying the formula to the dataset.
# These examples confirm that each tenfold increase in Ki lowers pKi by one unit.
for ki_example, expected_pki in ((1.0, 9.0), (10.0, 8.0), (100.0, 7.0)):
    assert math.isclose(9.0 - math.log10(ki_example), expected_pki, abs_tol=1e-12)
# Convert each nanomolar Ki to pKi and store it in a separate column.
# The original Ki stays unchanged, including for measurements sent to structure review.
measurements["pki"] = measurements["ki_nm"].map(lambda value: 9.0 - math.log10(value))
# Stop if any calculated target is missing, infinite, or otherwise not a finite number.
assert measurements["pki"].map(math.isfinite).all()
# Report the number of conversions and the lowest and highest observed pKi values.
# The displayed range is rounded for readability; stored values keep their precision.
print(f"pKi calculated for {len(measurements):,} selected activities.")
print(f"Observed pKi range: {measurements['pki'].min():.3f} to {measurements['pki'].max():.3f}")

pKi calculated for 4,081 selected activities.
Observed pKi range: 4.020 to 10.721


## 6. Review repeated structures and measurement disagreement

Every eligible structure retains links to all its activity, molecule, assay, and document IDs in the activity table. A one-unit pKi span corresponds to a tenfold Ki difference. The full measurement-level range triggers conflict review at the configured threshold, with a small absolute tolerance only at that boundary. Within-assay and across-assay ranges are context, not tests of independent experiments. Equal values do not prove that two distinct activity records are duplicates.

In [6]:
# Check the exact tenfold example that can calculate as slightly less than one pKi unit.
# The tolerance ensures this numerical rounding does not hide a boundary conflict.
boundary_example = (9.0 - math.log10(1.03)) - (9.0 - math.log10(10.3))
assert math.isclose(boundary_example, 1.0, rel_tol=0, abs_tol=CONFLICT_RANGE_ABS_TOL)
if CONFLICT_RANGE_PKI == 1.0:
    assert boundary_example >= CONFLICT_RANGE_PKI or math.isclose(
        boundary_example, CONFLICT_RANGE_PKI, rel_tol=0, abs_tol=CONFLICT_RANGE_ABS_TOL
    )

# Group structure-eligible activities by their full RDKit key, then summarize each assay separately.
# The median of those assay medians is the candidate target; the full measurement range determines conflict.
eligible = measurements[measurements["review_reason"].eq("")].copy()
summary_rows = []
for structure_key, group in eligible.groupby("rdkit_smiles", sort=True):
    assay_medians = group.groupby("assay_chembl_id")["pki"].median()
    assay_ranges = group.groupby("assay_chembl_id")["pki"].agg(lambda values: values.max() - values.min())
    pki_min = group["pki"].min()
    pki_max = group["pki"].max()
    pki_range = pki_max - pki_min
    overall_median = group["pki"].median()
    assay_median_target = assay_medians.median()
    summary_rows.append({
        "rdkit_smiles": structure_key,
        "n_measurements": len(group),
        "n_assays": group["assay_chembl_id"].nunique(),
        "n_documents": group["document_chembl_id"].nunique(dropna=True),
        "n_molecule_ids": group["molecule_chembl_id"].nunique(),
        "pki_min": pki_min,
        "pki_max": pki_max,
        "pki_range": pki_range,
        "pki_overall_median": overall_median,
        "max_within_assay_pki_range": assay_ranges.max(),
        "assay_median_pki_range": assay_medians.max() - assay_medians.min(),
        "pki_target_candidate": assay_median_target,
        "target_minus_overall_median": assay_median_target - overall_median,
        "measurement_conflict": bool(
            pki_range >= CONFLICT_RANGE_PKI
            or math.isclose(pki_range, CONFLICT_RANGE_PKI, rel_tol=0, abs_tol=CONFLICT_RANGE_ABS_TOL)
        ),
    })
summary_columns = [
    "rdkit_smiles", "n_measurements", "n_assays", "n_documents", "n_molecule_ids",
    "pki_min", "pki_max", "pki_range", "pki_overall_median",
    "max_within_assay_pki_range", "assay_median_pki_range",
    "pki_target_candidate", "target_minus_overall_median", "measurement_conflict",
]
# Build one summary row per eligible structure and assign final statuses to the activity rows.
# Every measurement belonging to a conflicting structure receives the same conflict-review reason.
structure_summary = pd.DataFrame(summary_rows, columns=summary_columns)
conflict_keys = set(structure_summary.loc[structure_summary["measurement_conflict"], "rdkit_smiles"])
measurements["status"] = "retained"
measurements.loc[measurements["review_reason"].ne(""), "status"] = "structure_review"
measurements.loc[measurements["rdkit_smiles"].isin(conflict_keys), "status"] = "measurement_conflict"
measurements.loc[measurements["status"].eq("measurement_conflict"), "review_reason"] = CONFLICT_REVIEW_REASON

print(f"{'Eligible structures':<34} | {len(structure_summary):>8,}")
print(f"{'Repeated eligible structures':<34} | {structure_summary['n_measurements'].gt(1).sum():>8,}")
print(f"{'Conflicting structures':<34} | {len(conflict_keys):>8,}")
print(f"{'Conflicting activity records':<34} | {measurements['status'].eq('measurement_conflict').sum():>8,}")
# Count conflicts caught only by the rounding tolerance, then report structure-review reasons.
# This shows whether the numerical safeguard actually changed any decisions in this run.
tolerance_only_keys = set(structure_summary.loc[
    structure_summary["measurement_conflict"] & structure_summary["pki_range"].lt(CONFLICT_RANGE_PKI),
    "rdkit_smiles",
])
tolerance_only_activities = int(measurements["rdkit_smiles"].isin(tolerance_only_keys).sum())
print(f"{'Tolerance-only conflict structures':<34} | {len(tolerance_only_keys):>8,}")
print(f"{'Tolerance-only conflict activities':<34} | {tolerance_only_activities:>8,}")
print("\nStructure review by reason:")
print(f"{'Reason':<38} | {'Activities':>10}")
structure_reason_counts = measurements.loc[
    measurements["status"].eq("structure_review"), "review_reason"
].value_counts().sort_index()
if structure_reason_counts.empty:
    print(f"{'(none)':<38} | {0:>10,}")
for reason, count in structure_reason_counts.items():
    print(f"{reason:<38} | {count:>10,}")
print(f"\nSensitivity only; the primary rule remains >= {CONFLICT_RANGE_PKI:g} "
      f"with {CONFLICT_RANGE_ABS_TOL:g} absolute tolerance:")
print(f"{'Range threshold':<20} | {'Flagged structures':>18}")
# Show how many structures would be flagged at three comparison thresholds.
# This sensitivity check only reports counts; it does not change the chosen policy or saved statuses.
for threshold in (0.5, 1.0, 1.5):
    flagged = structure_summary["pki_range"].map(
        lambda value: value >= threshold or math.isclose(
            value, threshold, rel_tol=0, abs_tol=CONFLICT_RANGE_ABS_TOL
        )
    )
    print(f"{threshold:<20.1f} | {flagged.sum():>18,}")
print("\nLargest disagreements (a compact source-linked view):")
# Display the five largest within-structure disagreements with example source identifiers.
# Those IDs and assay descriptions help a reviewer trace the measurements behind each disagreement.
for row in structure_summary.nlargest(5, "pki_range").itertuples():
    group = eligible[eligible["rdkit_smiles"].eq(row.rdkit_smiles)]
    ids = ",".join(str(value) for value in group["activity_id"].head(5))
    assays = ",".join(group["assay_chembl_id"].drop_duplicates().head(3))
    documents = ",".join(group["document_chembl_id"].dropna().drop_duplicates().head(3))
    description = str(group["assay_description"].iloc[0])[:100]
    print(f"  range {row.pki_range:5.2f} | n={row.n_measurements:>3} | activity IDs {ids}")
    print(f"    assays {assays} | documents {documents}")
    print(f"    example assay description: {description}")

Eligible structures                |    3,606
Repeated eligible structures       |      343
Conflicting structures             |       30
Conflicting activity records       |      142
Tolerance-only conflict structures |        0
Tolerance-only conflict activities |        0

Structure review by reason:
Reason                                 | Activities
multiple_components_review             |         34

Sensitivity only; the primary rule remains >= 1 with 1e-12 absolute tolerance:
Range threshold      | Flagged structures
0.5                  |                 51
1.0                  |                 30
1.5                  |                 14

Largest disagreements (a compact source-linked view):
  range  3.85 | n=  9 | activity IDs 2005089,2096572,2525266,3198572,6161490
    assays CHEMBL901032,CHEMBL950088,CHEMBL1002989 | documents CHEMBL1138302,CHEMBL1142334,CHEMBL1138444
    example assay description: Displacement of [3H]WIN-552122 from human recombinant CB2 receptor expresse

## 7. Keep one target per retained structure

For each retained structure, the candidate target above is the median of its assay-level median pKi values. This reduces the influence of an assay with many activity rows, while the overall activity-level median remains a diagnostic. A one-measurement structure keeps that measurement's pKi. Shared documents, assay-system differences, and unrecognized repeated experiments may still affect the result.

In [7]:
# Choose the final structure-level columns: a structure key, target, and supporting evidence counts.
# The minimum, maximum, range, and overall median retain context about the original measurements.
curated_columns = [
    "rdkit_smiles", "pki_target", "n_measurements", "n_assays", "n_documents",
    "n_molecule_ids", "pki_min", "pki_max", "pki_range", "pki_overall_median",
]
# Keep only structures without a conflict flag and promote their candidate value to pki_target.
# The tilde (~) reverses the Boolean flag, selecting rows where measurement_conflict is False.
retained_summary = structure_summary.loc[~structure_summary["measurement_conflict"]].copy()
curated = retained_summary.rename(columns={"pki_target_candidate": "pki_target"})[curated_columns]
curated = curated.sort_values("rdkit_smiles").reset_index(drop=True)
# Keep a separate activity-level table with every source field and curation decision.
# Sort activity and structure tables by stable keys so their saved row order is predictable.
measurement_columns = source_columns + [
    "pki", "rdkit_smiles", "component_count", "quality_flags", "status", "review_reason",
]
measurements = measurements[measurement_columns].sort_values("activity_id").reset_index(drop=True)
structure_summary = structure_summary.sort_values("rdkit_smiles").reset_index(drop=True)
# Confirm one finite target per structure and reconcile the underlying measurement counts.
# Summing n_measurements must recover the number of retained activity rows.
assert curated["rdkit_smiles"].is_unique
assert curated["pki_target"].map(math.isfinite).all()
assert curated["n_measurements"].sum() == measurements["status"].eq("retained").sum()
# Report the final sizes and how many structures rely on a single measurement.
# A single measurement supplies a target but cannot establish agreement between experiments.
print(f"{'Retained structures':<36} | {len(curated):>8,}")
print(f"{'Retained activity records':<36} | {measurements['status'].eq('retained').sum():>8,}")
print(f"{'Single-measurement retained structures':<36} | {curated['n_measurements'].eq(1).sum():>8,}")

Retained structures                  |    3,576
Retained activity records            |    3,905
Single-measurement retained structures |    3,263


## 8. Save the activity bridge, structure tables, and review workbook

A unique UTC run folder keeps this execution separate from earlier attempts. The activity CSV has exactly one row per selected input and is the bridge from each structure to every supporting record. The structure summary retains eligible conflicts for review; the curated CSV contains only retained keys. Review workbook strings are explicitly stored as text to prevent formula interpretation.

In [8]:
# Create a numbered output folder for curated data so previous runs remain intact.
# Reserve the manifest path now, but write the completion manifest only after the audit passes.
processed_parent = project_root / "data/processed/curation"
# Numbered runs keep previous results intact; dates are recorded in manifest.json.
processed_parent.mkdir(parents=True, exist_ok=True)
# Start above the highest saved run number after superseded runs are removed.
existing_run_numbers = [
    int(path.name[4:]) for path in processed_parent.glob("run_*")
    if path.is_dir() and path.name[4:].isdigit()
]
run_number = max(existing_run_numbers, default=0) + 1
while True:
    run_name = f"run_{run_number:02d}"
    output_folder = processed_parent / run_name
    try:
        output_folder.mkdir(exist_ok=False)
        break
    except FileExistsError:
        run_number += 1
measurement_path = output_folder / "measurement_decisions.csv"
summary_path = output_folder / "structure_summary.csv"
curated_path = output_folder / "curated_structures.csv"
review_path = output_folder / "review.xlsx"
curation_manifest_path = output_folder / "manifest.json"

# Save all activity decisions, all eligible structure summaries, and the retained modeling targets.
# Exclude the pandas row index and keep numeric precision rather than rounding the exported values.
measurements.to_csv(measurement_path, index=False)
structure_summary.to_csv(summary_path, index=False)
curated.to_csv(curated_path, index=False)

# Create a review workbook whose Summary sheet counts each final status and exclusion reason.
# The following sheets will retain the individual measurements behind those totals.
workbook = Workbook()
overview = workbook.active
overview.title = "Summary"
overview.append(["CB2 Ki curation run", run_name])
overview.append(["Status", "Activity records"])
for status in ("retained", "structure_review", "measurement_conflict"):
    overview.append([status, int(measurements["status"].eq(status).sum())])
overview.append([])
overview.append(["Review reason", "Activity records"])
for reason, count in sorted(measurements.loc[measurements["status"].ne("retained"), "review_reason"].value_counts().items()):
    overview.append([reason, int(count)])

# Give both review sheets the full activity columns plus three disagreement diagnostics.
# For conflicts, values[8] is rdkit_smiles in the declared column order and retrieves that structure summary.
review_diagnostic_columns = [
    "structure_pki_range", "max_within_assay_pki_range", "assay_median_pki_range",
]
review_columns = measurement_columns + review_diagnostic_columns
diagnostics_by_key = structure_summary.set_index("rdkit_smiles")
for sheet_name, status in (
    ("Structure review", "structure_review"),
    ("Measurement conflicts", "measurement_conflict"),
):
    sheet = workbook.create_sheet(sheet_name)
    sheet.append(review_columns)
    rows = measurements.loc[measurements["status"].eq(status), measurement_columns]
    for row in rows.itertuples(index=False, name=None):
        values = [None if pd.isna(value) else value for value in row]
        if status == "measurement_conflict":
            diagnostic = diagnostics_by_key.loc[values[8]]
            values.extend([
                diagnostic["pki_range"], diagnostic["max_within_assay_pki_range"],
                diagnostic["assay_median_pki_range"],
            ])
        else:
            values.extend([None, None, None])
        sheet.append(values)
        # Store source strings explicitly as text so Excel does not interpret them as formulas.
        # Freeze headers, enable filters, and widen columns to make manual review easier.
        for cell in sheet[sheet.max_row]:
            if isinstance(cell.value, str):
                cell.data_type = "s"
    sheet.freeze_panes = "A2"
    sheet.auto_filter.ref = sheet.dimensions
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for letter, width in {
        "A": 14, "B": 20, "C": 56, "D": 14, "E": 20, "F": 22,
        "G": 80, "H": 14, "I": 56, "J": 18, "K": 33, "L": 23, "M": 30,
        "N": 22, "O": 30, "P": 28,
    }.items():
        sheet.column_dimensions[letter].width = width
overview.freeze_panes = "A2"
overview.auto_filter.ref = overview.dimensions
overview.column_dimensions["A"].width = 42
overview.column_dimensions["B"].width = 38
for cell in overview[2]:
    cell.font = Font(bold=True)
# Save the workbook alongside the CSV files and print the destination.
# These files are not yet marked complete: the next cells reopen and audit their contents.
workbook.save(review_path)
print(f"Wrote four files to {output_folder.relative_to(project_root)}")
print("The run remains incomplete until the saved-file audit passes and the manifest is written.")


Wrote four files to data/processed/curation/run_05
The run remains incomplete until the saved-file audit passes and the manifest is written.


## 9. Audit the files read back from disk

The audit reloads the selected CSV, raw JSON, three output CSVs, and workbook. It reconstructs pKi and structure decisions from the saved source values, then recomputes each structure's counts, ranges, conflict status, and assay-median target. It compares full review rows, including reasons and numeric diagnostics, rather than relying on matching counts alone. An assertion failure stops execution before a completion manifest can be written.

In [9]:
# Reload the input and generated CSV files from disk, rather than trusting the export objects.
# Read fields as text so comparisons use deliberate numeric conversion and missing-value handling.
audit_input = pd.read_csv(selected_path, dtype=str, keep_default_na=False)
audit_measurements = pd.read_csv(measurement_path, dtype=str, keep_default_na=False)
audit_summary = pd.read_csv(summary_path, dtype=str, keep_default_na=False)
audit_curated = pd.read_csv(curated_path, dtype=str, keep_default_na=False)
# Reload the original activity and assay JSON files to establish the source of truth again.
# Check unique raw IDs and the expected column lists before comparing any saved values.
with activity_path.open(encoding="utf-8") as file:
    audit_raw_activities = json.load(file)
with assay_path.open(encoding="utf-8") as file:
    audit_raw_assays = json.load(file)
audit_raw_by_id = {row["activity_id"]: row for row in audit_raw_activities}
audit_assay_by_id = {row["assay_chembl_id"]: row for row in audit_raw_assays}
assert len(audit_raw_by_id) == len(audit_raw_activities)
assert len(audit_assay_by_id) == len(audit_raw_assays)
assert list(audit_measurements.columns) == measurement_columns
assert list(audit_summary.columns) == summary_columns
assert list(audit_curated.columns) == curated_columns
assert list(audit_input.columns) == source_columns

# Build input and output lookups by activity ID, checking that no duplicate IDs were collapsed.
# Matching ID sets then confirms that no selected measurement was lost or invented during export.
input_by_id = {int(row["activity_id"]): row for row in audit_input.to_dict("records")}
saved_by_id = {int(row["activity_id"]): row for row in audit_measurements.to_dict("records")}
assert len(input_by_id) == len(audit_input)
assert len(saved_by_id) == len(audit_measurements)
assert set(input_by_id) == set(saved_by_id)

# Compare all seven saved source fields with both the selected CSV and the original raw records.
# Use numeric comparisons for Ki and normalize only permitted blank or surrounding whitespace differences.
for activity_id, saved in saved_by_id.items():
    original = input_by_id[activity_id]
    raw = audit_raw_by_id[activity_id]
    assay = audit_assay_by_id[raw["assay_chembl_id"]]
    for field in source_columns:
        if field == "ki_nm":
            assert math.isclose(float(saved[field]), float(original[field]), rel_tol=0, abs_tol=1e-12), (activity_id, field)
            assert math.isclose(float(saved[field]), float(raw["standard_value"]), rel_tol=0, abs_tol=1e-12), (activity_id, field)
        elif field == "activity_id":
            assert int(saved[field]) == int(original[field]) == raw["activity_id"]
        else:
            assert blank_to_none(saved[field]) == blank_to_none(original[field]), (activity_id, field)
            source_value = assay.get("description") if field == "assay_description" else raw.get(field)
            assert blank_to_none(saved[field]) == blank_to_none(source_value), (activity_id, field)
    # Recalculate pKi directly from the raw Ki and compare it with the exported value.
    # A tight absolute tolerance permits floating-point differences without depending on the number's size.
    expected_pki = 9.0 - math.log10(float(raw["standard_value"]))
    assert math.isclose(float(saved["pki"]), expected_pki, rel_tol=0, abs_tol=1e-10), activity_id

# Record these two checks as passed only after all preceding assertions finish successfully.
# Any failed comparison stops the cell before this success message can be printed.
audit_results = {"source_values": True, "pki": True}
print(f"Saved source values and pKi verified for {len(saved_by_id):,} activities.")

Saved source values and pKi verified for 4,081 activities.


### Check structure assignments, review sheets, and aggregation

The structure check reparses distinct input strings and verifies both eligible keys and review assignments. Saved status groups must partition the input activity IDs exactly. The aggregation check starts from the saved activity table, whose source values and pKi were just verified. Numeric output comparisons use an absolute tolerance of `1e-10` and zero relative tolerance. The conflict decision is independently checked from raw Ki ratios with the configured boundary tolerance; the 1.03 and 10.3 nM example is an explicit regression check.

In [10]:
# Rebuild structure keys, review reasons, and quality flags from the saved input SMILES.
# This checks that exported structure decisions agree with a fresh application of the stated rules.
audit_structure = {}
for smiles in audit_input["canonical_smiles"].unique():
    flags = []
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=True)
    except (ValueError, RuntimeError) as error:
        mol = None
        flags.append("parse_exception_" + type(error).__name__)
    if mol is None:
        audit_structure[smiles] = (None, None, "parse_or_sanitization_failed", ";".join(flags))
        continue

    components = len(Chem.GetMolFrags(mol))
    if mol.GetNumAtoms() == 0:
        flags.append("zero_atoms")
    if any(atom.GetAtomicNum() == 0 for atom in mol.GetAtoms()):
        flags.append("wildcard_atom")
    if components > 1:
        flags.append("multiple_components")

    if "zero_atoms" in flags:
        reason = "zero_atoms"
    elif "wildcard_atom" in flags:
        reason = "wildcard_atom"
    elif "multiple_components" in flags:
        reason = "multiple_components_review"
    else:
        reason = ""
    key = None
    if not reason:
        try:
            key = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
            potential_stereo = Chem.FindPotentialStereo(mol)
            if any(info.specified == Chem.StereoSpecified.Unspecified for info in potential_stereo):
                flags.append("potential_unspecified_stereo")
        except (ValueError, RuntimeError) as error:
            reason = "structure_processing_failed"
            flags.append("processing_exception_" + type(error).__name__)
    audit_structure[smiles] = (key, components, reason, ";".join(flags))

# Check that the three allowed statuses account for every activity exactly once.
# Then compare each saved key, component count, reason, and quality flag with its recomputed result.
allowed_statuses = {"retained", "structure_review", "measurement_conflict"}
assert set(audit_measurements["status"]) <= allowed_statuses
status_ids = {}
for status in allowed_statuses:
    status_ids[status] = [int(value) for value in audit_measurements.loc[audit_measurements["status"].eq(status), "activity_id"]]
assert Counter(sum(status_ids.values(), [])) == Counter(input_by_id.keys())
for activity_id, row in saved_by_id.items():
    key, components, reason, expected_flags = audit_structure[row["canonical_smiles"]]
    assert row["quality_flags"] == expected_flags, (activity_id, "quality_flags")
    assert blank_to_none(row["rdkit_smiles"]) == key, activity_id
    saved_components = blank_to_none(row["component_count"])
    assert (int(float(saved_components)) if saved_components is not None else None) == components, activity_id
    if reason:
        assert row["status"] == "structure_review" and row["review_reason"] == reason, activity_id
    else:
        assert row["status"] != "structure_review", activity_id
        reparsed = Chem.MolFromSmiles(key, sanitize=True)
        assert reparsed is not None
        assert Chem.MolToSmiles(reparsed, canonical=True, isomericSmiles=True) == key

# Check unique structure rows and rebuild structure groups from the verified activity table.
# Recompute evidence counts, within-assay ranges, assay medians, and the final median of assay medians.
summary_by_key = {row["rdkit_smiles"]: row for row in audit_summary.to_dict("records")}
curated_by_key = {row["rdkit_smiles"]: row for row in audit_curated.to_dict("records")}
assert len(summary_by_key) == len(audit_summary)
assert len(curated_by_key) == len(audit_curated)
eligible_groups = {}
for row in saved_by_id.values():
    if row["status"] != "structure_review":
        eligible_groups.setdefault(row["rdkit_smiles"], []).append(row)
assert set(summary_by_key) == set(eligible_groups)

numeric_diagnostics = (
    "pki_min", "pki_max", "pki_range", "pki_overall_median",
    "max_within_assay_pki_range", "assay_median_pki_range",
    "pki_target_candidate", "target_minus_overall_median",
)
expected_retained_keys = set()
for key, rows in eligible_groups.items():
    values = pd.Series([float(row["pki"]) for row in rows])
    by_assay = {}
    for row in rows:
        by_assay.setdefault(row["assay_chembl_id"], []).append(float(row["pki"]))
    assay_medians = [float(pd.Series(group).median()) for group in by_assay.values()]
    within_ranges = [max(group) - min(group) for group in by_assay.values()]
    expected = {
        "n_measurements": len(rows),
        "n_assays": len(by_assay),
        "n_documents": len({row["document_chembl_id"] for row in rows if blank_to_none(row["document_chembl_id"]) is not None}),
        "n_molecule_ids": len({row["molecule_chembl_id"] for row in rows}),
        "pki_min": float(values.min()),
        "pki_max": float(values.max()),
        "pki_range": float(values.max() - values.min()),
        "pki_overall_median": float(values.median()),
        "max_within_assay_pki_range": max(within_ranges),
        "assay_median_pki_range": max(assay_medians) - min(assay_medians),
        "pki_target_candidate": float(pd.Series(assay_medians).median()),
    }
    expected["target_minus_overall_median"] = expected["pki_target_candidate"] - expected["pki_overall_median"]
    # Independently derive disagreement from log10(largest Ki / smallest Ki) using the raw values.
    # Check the conflict decision, every saved statistic, and the target for each retained structure.
    raw_ki_values = [float(audit_raw_by_id[int(row["activity_id"])]["standard_value"]) for row in rows]
    ratio_range = math.log10(max(raw_ki_values) / min(raw_ki_values))
    assert math.isclose(ratio_range, expected["pki_range"], rel_tol=0, abs_tol=1e-10), key
    conflict = ratio_range >= CONFLICT_RANGE_PKI or math.isclose(
        ratio_range, CONFLICT_RANGE_PKI, rel_tol=0, abs_tol=CONFLICT_RANGE_ABS_TOL
    )
    saved_summary = summary_by_key[key]
    for field in ("n_measurements", "n_assays", "n_documents", "n_molecule_ids"):
        assert int(saved_summary[field]) == expected[field], (key, field)
    for field in numeric_diagnostics:
        assert math.isclose(float(saved_summary[field]), expected[field], rel_tol=0, abs_tol=1e-10), (key, field)
    assert saved_summary["measurement_conflict"] == str(conflict), key
    assert all(row["status"] == ("measurement_conflict" if conflict else "retained") for row in rows), key
    if conflict:
        assert all(row["review_reason"] == CONFLICT_REVIEW_REASON for row in rows), key
    else:
        expected_retained_keys.add(key)
        saved_curated = curated_by_key[key]
        for field in ("n_measurements", "n_assays", "n_documents", "n_molecule_ids"):
            assert int(saved_curated[field]) == expected[field], (key, field)
        for field in ("pki_min", "pki_max", "pki_range", "pki_overall_median"):
            assert math.isclose(float(saved_curated[field]), expected[field], rel_tol=0, abs_tol=1e-10), (key, field)
        assert math.isclose(float(saved_curated["pki_target"]), expected["pki_target_candidate"], rel_tol=0, abs_tol=1e-10), key

# Verify that the curated file contains exactly the expected structures and supporting measurements.
# Repeat the tenfold boundary example to ensure rounding still triggers the intended conflict decision.
assert set(curated_by_key) == expected_retained_keys
assert all(math.isfinite(float(row["pki_target"])) for row in curated_by_key.values())
assert sum(int(row["n_measurements"]) for row in curated_by_key.values()) == len(status_ids["retained"])

tenfold_range = (9.0 - math.log10(1.03)) - (9.0 - math.log10(10.3))
assert math.isclose(tenfold_range, 1.0, rel_tol=0, abs_tol=CONFLICT_RANGE_ABS_TOL)
if CONFLICT_RANGE_PKI == 1.0:
    assert tenfold_range >= CONFLICT_RANGE_PKI or math.isclose(
        tenfold_range, CONFLICT_RANGE_PKI, rel_tol=0, abs_tol=CONFLICT_RANGE_ABS_TOL
    )
    assert math.log10(10.3 / 1.03) >= CONFLICT_RANGE_PKI

# Reopen the workbook and check its Summary title, run name, status totals, and review-reason counts.
# Compare these values with the audited activity rows, not just the workbook's own detailed sheets.
saved_workbook = load_workbook(review_path, data_only=False, read_only=False)
assert set(saved_workbook.sheetnames) == {"Summary", "Structure review", "Measurement conflicts"}
overview = saved_workbook["Summary"]
assert overview["A1"].value == "CB2 Ki curation run"
assert overview["B1"].value == review_path.parent.name
assert (overview["A2"].value, overview["B2"].value) == ("Status", "Activity records")
for row_number, status in enumerate(("retained", "structure_review", "measurement_conflict"), start=3):
    assert overview.cell(row_number, 1).value == status
    assert overview.cell(row_number, 2).value == len(status_ids[status])
assert overview["A6"].value is None and overview["B6"].value is None
assert (overview["A7"].value, overview["B7"].value) == ("Review reason", "Activity records")
expected_reasons = Counter(
    row["review_reason"] for row in saved_by_id.values() if row["status"] != "retained"
)
saved_reasons = [
    (reason, count) for reason, count in overview.iter_rows(min_row=8, max_col=2, values_only=True)
    if reason is not None or count is not None
]
assert saved_reasons == sorted(expected_reasons.items())
assert overview.freeze_panes == "A2" and overview.auto_filter.ref is not None
# Check each detailed review sheet against the verified CSV, including values, reasons, and diagnostics.
# Also check that text stayed text, headers and filters are present, and review IDs match without duplicates.
for sheet_name, status in (("Structure review", "structure_review"), ("Measurement conflicts", "measurement_conflict")):
    sheet = saved_workbook[sheet_name]
    assert [cell.value for cell in sheet[1]] == review_columns
    assert sheet.freeze_panes == "A2" and sheet.auto_filter.ref is not None
    workbook_ids = []
    for excel_row in sheet.iter_rows(min_row=2):
        values = [cell.value for cell in excel_row]
        if all(value is None for value in values):
            continue
        record = dict(zip(review_columns, values))
        activity_id = int(record["activity_id"])
        workbook_ids.append(activity_id)
        saved = saved_by_id[activity_id]
        assert saved["status"] == status
        for field, cell in zip(measurement_columns, excel_row):
            value = record[field]
            if field in ("activity_id", "component_count"):
                expected_value = blank_to_none(saved[field])
                assert (int(value) if value is not None else None) == (int(float(expected_value)) if expected_value is not None else None), (activity_id, field)
            elif field in ("ki_nm", "pki"):
                assert math.isclose(float(value), float(saved[field]), rel_tol=0, abs_tol=1e-10), (activity_id, field)
            else:
                assert blank_to_none(value) == blank_to_none(saved[field]), (activity_id, field)
                if isinstance(value, str):
                    assert cell.data_type == "s", (activity_id, field)
        for field, summary_field in (
            ("structure_pki_range", "pki_range"),
            ("max_within_assay_pki_range", "max_within_assay_pki_range"),
            ("assay_median_pki_range", "assay_median_pki_range"),
        ):
            expected_value = summary_by_key[saved["rdkit_smiles"]][summary_field] if status == "measurement_conflict" else None
            if expected_value is None:
                assert record[field] is None, (activity_id, field)
            else:
                assert math.isclose(float(record[field]), float(expected_value), rel_tol=0, abs_tol=1e-10), (activity_id, field)
    assert Counter(workbook_ids) == Counter(status_ids[status])
saved_workbook.close()

# Mark the remaining audit checks as passed only after all comparisons above succeed.
# A failure interrupts execution before this block and before a completion manifest can be written.
audit_results.update({
    "structure": True, "quality_flags": True, "aggregation": True,
    "workbook_summary": True, "saved_files": True,
})
# Collect and print the audited totals, distinguishing activity rows from unique structures.
# Include single-measurement structures and tolerance-only exclusions so the results are interpretable.
audit_counts = {
    "input_activities": len(audit_input),
    "retained_activities": len(status_ids["retained"]),
    "structure_review_activities": len(status_ids["structure_review"]),
    "conflict_activities": len(status_ids["measurement_conflict"]),
    "eligible_structures": len(summary_by_key),
    "conflict_structures": len(summary_by_key) - len(curated_by_key),
    "final_curated_structures": len(curated_by_key),
    "single_measurement_structures": sum(int(row["n_measurements"]) == 1 for row in curated_by_key.values()),
    "tolerance_only_conflict_structures": len(tolerance_only_keys),
    "tolerance_only_conflict_activities": tolerance_only_activities,
}
print("AUDIT SUMMARY")
for label, count in audit_counts.items():
    print(f"  {label.replace('_', ' '):<34} | {count:>8,}")
for label, passed in audit_results.items():
    print(f"  {label.replace('_', ' '):<34} | {'PASS' if passed else 'FAIL':>8}")

AUDIT SUMMARY
  input activities                   |    4,081
  retained activities                |    3,905
  structure review activities        |       34
  conflict activities                |      142
  eligible structures                |    3,606
  conflict structures                |       30
  final curated structures           |    3,576
  single measurement structures      |    3,263
  tolerance only conflict structures |        0
  tolerance only conflict activities |        0
  source values                      |     PASS
  pki                                |     PASS
  structure                          |     PASS
  quality flags                      |     PASS
  aggregation                        |     PASS
  workbook summary                   |     PASS
  saved files                        |     PASS


## 10. Record provenance only after the audit passes

The completion manifest ties this run to its exact source snapshot, policy, software versions, schemas, counts, and SHA-256 file hashes. Its creation is the completion marker, so an interrupted or failed run remains visibly incomplete. The manifest does not hash itself.

In [11]:
# Require every audit result to pass before this run receives its completion record.
assert all(audit_results.values()), "The completion manifest requires every audit check."
# Calculate a SHA-256 digest, a content identifier used to detect later changes to file bytes.
# Read in chunks so hashing a large raw JSON file does not require loading it all into memory.
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Record the current git commit and whether git reports uncommitted changes.
# This helps connect the data run with the state of the project code used to produce it.
git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=project_root, capture_output=True, text=True, check=False
).stdout.strip() or None
git_dirty = bool(subprocess.run(
    ["git", "status", "--porcelain"], cwd=project_root, capture_output=True, text=True, check=False
).stdout.strip())
# List the exact input and generated files to hash, excluding the manifest itself.
# A manifest cannot straightforwardly contain its own hash because writing that hash changes its bytes.
hashed_paths = (
    selected_path, selection_manifest_path, acquisition_manifest_path, activity_path, assay_path,
    measurement_path, summary_path, curated_path, review_path,
)
# Assemble the run history: source paths, hashes, software versions, policies, counts, and output columns.
# Keep paths relative to the repository so this record remains useful when the project is moved.
curation_manifest = {
    "run_started_at_utc": run_started_at_utc,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": NOTEBOOK_REL,
    "target_chembl_id": acquisition_manifest["target_chembl_id"],
    "target_organism": acquisition_manifest["organism"],
    "uniprot": acquisition_manifest["uniprot"],
    "chembl_version": acquisition_manifest["chembl_version"],
    "selection_manifest": selection_manifest_path.relative_to(project_root).as_posix(),
    "selected_csv": selected_path.relative_to(project_root).as_posix(),
    "source_raw_folder": raw_folder.relative_to(project_root).as_posix(),
    "acquisition_manifest": acquisition_manifest_path.relative_to(project_root).as_posix(),
    "output_folder": output_folder.relative_to(project_root).as_posix(),
    "file_sha256": {path.relative_to(project_root).as_posix(): sha256_file(path) for path in hashed_paths},
    "git": {"commit": git_commit, "working_tree_dirty": git_dirty},
    "software_versions": {
        "python": platform.python_version(), "rdkit": rdkit.__version__,
        "pandas": pd.__version__, "openpyxl": __import__("openpyxl").__version__,
    },
    "policies": {
        "version": "proposed_v1",
        "structure_normalization": "RDKit parse and sanitize; canonical isomeric SMILES only",
        "structure_key": "full RDKit canonical isomeric SMILES in rdkit_smiles",
        "multiple_components": "quarantine complete representation; no salt or largest-fragment selection",
        "charge_tautomers_isotopes_stereo": "preserve represented information; no neutralization, tautomer normalization, or stereo/isotope removal",
        "structure_primary_reason_order": ["parse_or_sanitization_failed", "zero_atoms", "wildcard_atom", "multiple_components_review", "structure_processing_failed"],
        "conflict_threshold_pki": CONFLICT_RANGE_PKI,
        "conflict_absolute_tolerance_pki": CONFLICT_RANGE_ABS_TOL,
        "conflict_review_reason": CONFLICT_REVIEW_REASON,
        "conflict_operator": ">= or within absolute tolerance of threshold",
        "conflict_basis": "full measurement-level pKi range within rdkit_smiles",
        "conflict_action": "all activities for a conflicting structure go to review",
        "assay_grouping_keys": ["rdkit_smiles", "assay_chembl_id"],
        "target_aggregation": "median pKi within each structure-assay group, then median of those assay medians",
        "single_measurement_structure": "retain with n_measurements=1",
        "missing_document_id": "preserve as missing; exclude only from distinct document count",
    },
    "counts": audit_counts,
    "final_status_counts": {status: len(ids) for status, ids in status_ids.items()},
    "review_reason_activity_counts": {
        reason: int(count) for reason, count in audit_measurements.loc[
            audit_measurements["status"].ne("retained"), "review_reason"
        ].value_counts().items()
    },
    "output_schemas": {
        measurement_path.name: measurement_columns,
        summary_path.name: summary_columns,
        curated_path.name: curated_columns,
        review_path.name: {"Structure review": review_columns, "Measurement conflicts": review_columns},
    },
    "audit": {"absolute_tolerance": 1e-10, "relative_tolerance": 0, "results": audit_results},
}
# Write the JSON completion manifest only after the data files have passed the audit.
# Confirm that it exists and print the completed run location for the next notebook.
with curation_manifest_path.open("w", encoding="utf-8") as file:
    json.dump(curation_manifest, file, indent=2, ensure_ascii=False)
assert curation_manifest_path.is_file()
print(f"Completed curation run: {output_folder.relative_to(project_root)}")
print(f"Completion manifest: {curation_manifest_path.name}")

Completed curation run: data/processed/curation/run_05
Completion manifest: manifest.json


## Interpretation and next stage

The audit summary above gives the observed numbers of retained activities and structures, plus the measurements and complete structures set aside for review. A pKi conflict is a disagreement among measurements of **one** structure; an activity cliff would compare **different, similar** structures and is not identified here. The range rule is more likely to flag structures with many measurements, and assay or document IDs do not establish independent experiments.

This version keeps protonation and tautomer forms separate and sets aside multicomponent representations; these may warrant later chemical review. The curated values are traceable summaries of selected ChEMBL records, not a certification that the original experiments are correct. The next stage can build fingerprints or molecular graphs and plan data splits. Identical structures must stay in one train/validation/test subset; close analogs, unresolved protonation or tautomer variants, and shared assays or documents also need attention when designing splits. Any feature scaling or fitted preprocessing belongs after splitting and must be fitted on the training subset only.